# SKILL EXRACTING MODEL
Install the dependencies

In [ ]:
!pip install transformers datasets seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=895473a2b65c7ddc4ba37078180df77c64da7ed8c4251fc944063a99b82ad1f7
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


Load the dataset

In [2]:
import pandas as pd
import csv
import sys

df = pd.read_csv("/content/sample_data/cleaned_dataset.csv")

In [3]:
df.columns

Index(['topic', 'title', 'tokens', 'ner_tags', 'extracted_skills',
       'num_tokens', 'num_skills_found', 'topic_clean', 'title_clean'],
      dtype='object')

In [ ]:
from transformers import BertTokenizerFast
tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")

# Step 1: Convert tokens column to string if it's a list
import ast

def clean_tokens(token_cell):
    # If the cell is a string representation of a list, convert to list
    if isinstance(token_cell, str) and token_cell.startswith('[') and token_cell.endswith(']') :
        try:
            token_cell = ast.literal_eval(token_cell)
        except:
            pass
    # Ensure it's a list
    if isinstance(token_cell, list):
        # Remove quotes/brackets, keep only strings
        token_cell = [str(t).replace('"', '').replace("'", "").strip() for t in token_cell]
    return token_cell

# Step 2: Use only tokens with labels for training
df['tokens_clean'] = df['tokens'].apply(clean_tokens)

# # Step 3: Convert tokens_clean to string for BERT tokenizer (if needed)
# df['tokens_text'] = df['tokens_clean'].apply(lambda x: ' '.join(x))

# Step 4: Tokenize for BERT
tokenized_encodings = tokenizer(
    df['tokens_clean'].tolist(),
    truncation=True,
    padding=True,
    is_split_into_words=True  # important for label alignment
)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

In [ ]:
df['tokens_clean']

,tokens_clean
0,"[Nordic, founders, are, taking, bigger, swings..."
1,"[Here, are, the, 49, US, AI, startups, that, h..."
2,"[JustiGuide, wants, to, use, AI, to, help, peo..."
3,"[OpenAI, claims, teen, circumvented, safety, f..."
4,"[Redwood, Materials, reportedly, cuts, 5, of, ..."
...,...
687,"[For, Crypto, Holders, ,, DeFi, Lending, Rates..."
688,"[This, Stock, Is, Down, 44, in, 2025, but, Leg..."
689,"[LINK, FOREX, Unveils, Strategic, Plan, for, t..."
690,"[Molten, Ventures, eyes, selling, more, Revolu..."


Turn the ner_tags to string->list

In [ ]:
print(df['ner_tags'].iloc[0])


["O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "B-SKILL", "I-SKILL", "O", "B-SKILL", "O", "B-SKILL", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "B-SKILL", "O", "B-SKILL", "O", "O", "O", "O"]


In [ ]:
import ast

df['ner_tags'] = df['ner_tags'].apply(lambda x: ast.literal_eval(x))


In [ ]:
print(type(df['ner_tags'].iloc[0]))
print(df['ner_tags'].iloc[0][:10])


<class 'list'>
['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']


In [ ]:
# Suppose your ner_tags contain these unique labels
unique_labels = df['ner_tags'].explode().unique()  # flatten lists in ner_tags column
label_list = list(unique_labels)

# Create mappings
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for label, i in label2id.items()}

print(label_list)
print(label2id)


['O', 'B-SKILL', 'I-SKILL']
{'O': 0, 'B-SKILL': 1, 'I-SKILL': 2}


In [ ]:
def align_labels_with_tokens(labels, word_ids):
    aligned_labels = []
    for word_id in word_ids:
        if word_id is None:
            aligned_labels.append(-100)
        else:
            aligned_labels.append(labels[word_id])
    return aligned_labels

aligned_labels = []

for i in range(len(df)):
    labels = [label2id[l] for l in df['ner_tags'].iloc[i]]  # convert to numeric
    word_ids = tokenized_encodings.word_ids(batch_index=i)
    aligned = align_labels_with_tokens(labels, word_ids)
    aligned_labels.append(aligned)


In [ ]:
import torch
from torch.utils.data import Dataset

class SkillsTokenDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        # Convert each encoding to a tensor
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        # Add labels
        item['labels'] = torch.tensor(self.labels[idx])
        return item


In [ ]:
dataset = SkillsTokenDataset(tokenized_encodings, aligned_labels)


In [ ]:
df.to_csv("preprocessed_data.csv", index=False)

Break the data into train, validation and test data

In [ ]:
from torch.utils.data import DataLoader, random_split

# Suppose dataset length
dataset_size = len(dataset)
train_size = int(0.7 * dataset_size)   # 70% train
val_size = int(0.15 * dataset_size)    # 15% validation
test_size = dataset_size - train_size - val_size  # 15% test

generator = torch.Generator().manual_seed(42)  # fixed seed for reproducibility
train_dataset, val_dataset, test_dataset = random_split(dataset, [train_size, val_size, test_size], generator=generator)

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)


In [ ]:
from torch.nn import CrossEntropyLoss
from transformers import BertForTokenClassification
from torch.optim import AdamW
import torch

num_labels = len(label_list)

model = BertForTokenClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

optimizer = AdamW(model.parameters(), lr=5e-5)


epochs = 3
model.train()

for epoch in range(epochs):
    total_loss = 0
    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{epochs} - Avg Loss: {avg_loss:.4f}")

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1/3 - Avg Loss: 0.2520
Epoch 2/3 - Avg Loss: 0.1366
Epoch 3/3 - Avg Loss: 0.0957


Define regex patterns

In [ ]:
def ner_predict(ner_model, tokenizer, text):
    ner_model.eval()
    tokens = text.split()

    encoding = tokenizer(
        tokens,
        is_split_into_words=True,
        return_tensors="pt",
        truncation=True,
        max_length=128
    )

    # Compute word_ids before moving to device
    word_ids = encoding.word_ids(batch_index=0)

    # Move tensors to device
    device = next(ner_model.parameters()).device
    encoding = {k: v.to(device) for k, v in encoding.items()}

    # Predict
    with torch.no_grad():
        outputs = ner_model(**encoding)
        predictions = torch.argmax(outputs.logits, dim=2)

    # Map predictions to original tokens
    predicted_labels = []
    previous_word_idx = None
    for i, word_idx in enumerate(word_ids):
        if word_idx is None:
            continue
        if word_idx != previous_word_idx:
            predicted_labels.append(predictions[0][i].item())
            previous_word_idx = word_idx

    # Map to label names
    id2label_map = {0: "O", 1: "B-SKILL", 2: "I-SKILL"}

    skills = []
    current_skill = []
    for token, label_id in zip(tokens, predicted_labels):
        label = id2label_map[label_id]
        if label == "B-SKILL":
            if current_skill:
                skills.append(" ".join(current_skill))
            current_skill = [token]
        elif label == "I-SKILL":
            if current_skill:
                current_skill.append(token)
        else:
            if current_skill:
                skills.append(" ".join(current_skill))
                current_skill = []

    if current_skill:
        skills.append(" ".join(current_skill))

    return skills


In [ ]:
import re
import string

# =============================
# Regex patterns for skills
# =============================
patterns = [
    r"(\b\w{2,}\b)\s+programming\s+language",  # words >=2 chars before "programming language"
    r"experience\s+with\s+(\b[\w\+\#]{2,}\b)",
    r"proficient\s+in\s+(\b[\w\+\#]{2,}\b)",
    r"(\b[\w\+\#]{2,}\b)\s+framework",
    r"(\b[\w\+\#]{2,}\b)\s+developer[s]?",
    r"knowledge\s+of\s+(\b[\w\+\#]{2,}\b)",
    r"using\s+(\b[\w\+\#]{2,}\b)\s+(?:for|to)",
    r"built\s+with\s+(\b[\w\+\#]{2,}\b)",
]

stopwords = {
    "the","this","our","your","their","that","these","those",
    "with","for","and","are","is","among","many","features",
    "developers"
}

# =============================
# Hybrid skill detector
# =============================
def hybrid_detect(text, ner_model=None, tokenizer=None):
    detected_skills = set()

    # ---- 1) Use NER model predictions ----
    if ner_model and tokenizer:
        raw_skills = ner_predict(ner_model, tokenizer, text)
        for s in raw_skills:
            detected_skills.add(s)

    # ---- 2) Use regex patterns ----
    for pattern in patterns:
        matches = re.findall(pattern, text, flags=re.IGNORECASE)
        for m in matches:
            if isinstance(m, tuple):
                m = m[0]
            m_clean = m.strip(string.punctuation)
            if len(m_clean) >= 2 and m_clean.lower() not in stopwords:
                detected_skills.add(m_clean)

    return sorted(detected_skills)

# =============================
# Example usage
# =============================
test_texts = [
    "Developers are adopting Furina programming language for AI workloads",
    "Experience with J and C++ frameworks",
    "Proficient in PyTorch and TensorFlow"
]

for text in test_texts:
    skills = hybrid_detect(text, ner_model=model, tokenizer=tokenizer)
    print(f"Text: {text}")
    print("Detected skills:", skills)
    print("-"*50)


Text: Developers are adopting Furina programming language for AI workloads
Detected skills: ['AI', 'Furina']
--------------------------------------------------
Text: Experience with J and C++ frameworks
Detected skills: ['C++', 'J']
--------------------------------------------------
Text: Proficient in PyTorch and TensorFlow
Detected skills: ['PyTorch', 'TensorFlow']
--------------------------------------------------


In [ ]:
model.save_pretrained("skill_ner_bert")
tokenizer.save_pretrained("skill_ner_bert")


('skill_ner_bert/tokenizer_config.json',
 'skill_ner_bert/special_tokens_map.json',
 'skill_ner_bert/vocab.txt',
 'skill_ner_bert/added_tokens.json',
 'skill_ner_bert/tokenizer.json')

In [ ]:
from sklearn.metrics import classification_report

all_true_labels = []
all_pred_labels = []

model.eval()
with torch.no_grad():
    for batch in val_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        predictions = torch.argmax(outputs.logits, dim=-1)

        for i in range(labels.shape[0]):  # iterate over batch
            true_seq = labels[i].cpu().tolist()
            pred_seq = predictions[i].cpu().tolist()
            word_ids = batch["input_ids"][i]  # can be used if doing word-level alignment

            # Align special tokens to ignore them
            for t, p in zip(true_seq, pred_seq):
                if t != -100:  # -100 is usually used for ignored tokens
                    all_true_labels.append(id2label[t])
                    all_pred_labels.append(id2label[p])


In [ ]:
from seqeval.metrics import precision_score, recall_score, f1_score

precision = precision_score(all_true_labels, all_pred_labels)
recall = recall_score(all_true_labels, all_pred_labels)
f1 = f1_score(all_true_labels, all_pred_labels)

print(f"Achieving {precision*100:.2f}% precision and {recall*100:.2f}% recall")


In [ ]:
# Print detailed classification report
print(classification_report(all_true_labels, all_pred_labels, digits=4))


              precision    recall  f1-score   support

     B-SKILL     0.6967    0.5375    0.6068      2679
     I-SKILL     0.5813    0.3381    0.4275       846
           O     0.9528    0.9790    0.9657     35083

    accuracy                         0.9343     38608
   macro avg     0.7436    0.6182    0.6667     38608
weighted avg     0.9269    0.9343    0.9290     38608



In [ ]:
import torch
import string

# Make sure model and tokenizer are already loaded
# model -> trained BertForTokenClassification
# tokenizer -> BertTokenizerFast
# id2label -> {0: "O", 1: "B-SKILL", 2: "I-SKILL"}

def merge_subwords(tokens, labels):
    """Merge BERT subwords and filter punctuation/numbers"""
    merged_tokens = []
    merged_labels = []

    word = ""
    label = None
    skip_tokens = {"[CLS]", "[SEP]", "[PAD]"}

    for t, l in zip(tokens, labels):
        if t in skip_tokens:
            continue
        if t.startswith("##"):
            word += t[2:]
        else:
            if word:
                merged_tokens.append(word)
                merged_labels.append(label)
            word = t
            label = l
    if word:
        merged_tokens.append(word)
        merged_labels.append(label)

    # filter punctuation/numbers (optional)
    final = [(w, l) for w, l in zip(merged_tokens, merged_labels)
             if not all(c in string.punctuation + string.digits for c in w)]
    return final

# -----------------------------
# Evaluate on first batch from val_loader
# -----------------------------
model.eval()
with torch.no_grad():
    batch = next(iter(val_loader))  # first batch
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)

    # Forward pass
    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    predictions = torch.argmax(outputs.logits, dim=-1)

    # Convert to readable labels
    pred_labels = [
        [id2label[id.item()] for id in seq]
        for seq in predictions
    ]

    # Take the first sequence in the batch for demonstration
    input_ids_seq = input_ids[0].cpu().tolist()
    tokens = tokenizer.convert_ids_to_tokens(input_ids_seq)
    labels = pred_labels[0]

    # Merge subwords and clean output
    cleaned_result = merge_subwords(tokens, labels)

# -----------------------------
# Print final token -> predicted label
# -----------------------------
for token, label in cleaned_result:
    print(f"{token} -> {label}")


adda -> O
data -> B-SKILL
breach -> I-SKILL
personal -> O
data -> O
of -> O
over -> O
lakh -> O
users -> O
hacked -> O
and -> O
posted -> O
online -> O
read -> O
all -> O
details -> O
here -> O
hackers -> O
could -> O
use -> O
their -> O
name -> O
to -> O
trigger -> O
phishing -> B-SKILL
attacks -> O
whats -> O
more -> O
is -> O
that -> O
the -> O
user -> O
credentials -> O
that -> O
surface -> O
from -> O
one -> O
data -> B-SKILL
breach -> I-SKILL
can -> O
also -> O
be -> O
utilised -> O
by -> O
other -> O
hackers -> O
to -> O
attempt -> O
to -> O
log -> O
in -> O
to -> O
user -> O
accounts -> O
on -> O
other -> O
platforms -> O
to -> O
carry -> O
out -> O
credentials -> B-SKILL
stuffing -> I-SKILL
attack -> O
the -> O
data -> O
has -> O
been -> O
leaked -> O
just -> O
a -> O
few -> O
days -> O
after -> O
the -> O
government -> O
of -> O
india -> O
launched -> O
the -> O
digital -> O
personal -> O
data -> I-SKILL
protection -> I-SKILL
dpdp -> B-SKILL
rules -> O
until -> O
now -> O
add

In [ ]:
# Keep only predicted skills
predicted_skills = [token for token, label in cleaned_result if label != "O"]
print(predicted_skills)


['data', 'breach', 'phishing', 'data', 'breach', 'credentials', 'stuffing', 'data', 'protection', 'dpdp', 'data', 'breach']


Test the model

In [ ]:
model.eval()  # set to evaluation mode
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=-1)

        # Collect predictions and true labels
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())


In [ ]:
true_labels_str = []
pred_labels_str = []

for seq_preds, seq_labels in zip(all_preds, all_labels):
    pred_seq = []
    true_seq = []
    for p, l in zip(seq_preds, seq_labels):
        if l != -100:  # ignore padding / special tokens
            pred_seq.append(id2label[p])
            true_seq.append(id2label[l])
    pred_labels_str.append(pred_seq)
    true_labels_str.append(true_seq)


In [ ]:
from seqeval.metrics import classification_report

print(classification_report(true_labels_str, pred_labels_str))


              precision    recall  f1-score   support

       SKILL       0.61      0.55      0.58      2440

   micro avg       0.61      0.55      0.58      2440
   macro avg       0.61      0.55      0.58      2440
weighted avg       0.61      0.55      0.58      2440



Test with a sentence

In [ ]:
# Example sentence
sentence = "go language is new programming language"

# Step 1: Tokenize the sentence
encoding = tokenizer(
    sentence.split(),             # split sentence into words
    truncation=True,
    padding='max_length',         # optional: pad to max_length
    is_split_into_words=True,     # important for NER alignment
    return_tensors="pt"
)

input_ids = encoding["input_ids"].to(device)
attention_mask = encoding["attention_mask"].to(device)

# Step 2: Model prediction
model.eval()
with torch.no_grad():
    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    predictions = torch.argmax(outputs.logits, dim=-1)  # shape: [1, seq_len]

# Step 3: Align tokens with predictions
tokens = tokenizer.convert_ids_to_tokens(input_ids[0])
pred_labels = [id2label[p.item()] for p in predictions[0]]

# Step 4: Merge subwords and skip special tokens
import string

def merge_subwords(tokens, labels):
    cleaned = []
    skip_tokens = {"[CLS]", "[SEP]", "[PAD]"}
    current_token = ""
    current_label = ""
    for token, label in zip(tokens, labels):
        if token in skip_tokens:
            continue
        if token.startswith("##"):  # continuation of previous token
            current_token += token[2:]
        else:
            if current_token:
                cleaned.append((current_token, current_label))
            current_token = token
            current_label = label
    if current_token:
        cleaned.append((current_token, current_label))
    return cleaned

result = merge_subwords(tokens, pred_labels)

# Step 5: Print results
for token, label in result:
    print(f"{token} -> {label}")


go -> B-SKILL
language -> O
is -> O
new -> O
programming -> O
language -> O


In [ ]:
from os import read
import pandas as pd
df=pd.read_csv("/content/preprocessed_data.csv")


# Previous Build Codes for later use

In [ ]:
# from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

# # Example: T5 model fine-tuned for grammar correction
# model_name = "prithivida/grammar_error_correcter_v1"
# tokenizer = AutoTokenizer.from_pretrained(model_name)
# model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# def correct_text(text):
#     inputs = tokenizer.encode("gec: " + text, return_tensors="pt", truncation=True)
#     outputs = model.generate(inputs, max_length=512)
#     corrected_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
#     return corrected_text


In [ ]:
# df['summary_corrected'] = df['summary'].apply(correct_text)
# print(df[['summary', 'summary_corrected']].head())


In [ ]:
# df["text"] = df["title"].str.lower() + ". " + df["summary_corrected"]

In [ ]:
# import ast

# # Convert string to list
# df['skills_filtered'] = df['skills_filtered'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)



In [ ]:
# df.to_csv("preprocessed_data.csv", index=False)

In [ ]:
# import nltk
# nltk.download('punkt')
# from nltk.tokenize import word_tokenize

# def create_ner_labels(text, skill_list):
#     tokens = word_tokenize(text)
#     tokens_lower = [t.lower() for t in tokens]
#     labels = ["O"] * len(tokens)

#     for skill in skill_list:
#         skill_tokens = word_tokenize(skill)
#         skill_tokens_lower = [t.lower() for t in skill_tokens]
#         L = len(skill_tokens_lower)
#         if L == 0:
#             continue
#         for i in range(len(tokens_lower) - L + 1):
#             if tokens_lower[i:i + L] == skill_tokens_lower:
#                 labels[i] = "B-SKILL"
#                 for j in range(1, L):
#                     labels[i + j] = "I-SKILL"
#     return tokens, labels

# df["tokens"], df["ner_tags"] = zip(*df.apply(lambda r: create_ner_labels(r["text"], r["skills_filtered"]), axis=1))


In [ ]:
from transformers import BertTokenizerFast

tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")

# Map labels to numbers
label_list = ["O", "B-SKILL", "I-SKILL"]
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for label, i in label2id.items()}

# Tokenize and align labels
def tokenize_and_align_labels(tokens, ner_tags):
    encoding = tokenizer(tokens, is_split_into_words=True, truncation=True, padding="max_length", max_length=128)
    labels = []
    word_ids = encoding.word_ids()  # map tokens to original words
    previous_word_idx = None
    for word_idx in word_ids:
        if word_idx is None:
            labels.append(-100)  # ignore padding
        elif word_idx != previous_word_idx:
            labels.append(label2id[ner_tags[word_idx]])  # first token of word
        else:
            # same word as previous token (for wordpieces)
            if ner_tags[word_idx].startswith("B-"):
                labels.append(label2id["I-SKILL"])  # inside the skill
            else:
                labels.append(label2id[ner_tags[word_idx]])
        previous_word_idx = word_idx
    return encoding, labels

In [ ]:

tokenized_encodings = []
aligned_labels = []

for tokens, ner_tags in zip(df["tokens"], df["ner_tags"]):
    encoding, labels = tokenize_and_align_labels(tokens, ner_tags)
    tokenized_encodings.append(encoding)
    aligned_labels.append(labels)

In [ ]:
df.to_csv("data_tokenized.csv", index=False)

In [ ]:
import torch
from torch.utils.data import Dataset

class SkillsTokenDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

# Convert list of encodings to dict of lists
all_input_ids = [e["input_ids"] for e in tokenized_encodings]
all_attention_mask = [e["attention_mask"] for e in tokenized_encodings]

tokenized_encodings_dict = {
    "input_ids": all_input_ids,
    "attention_mask": all_attention_mask
}

dataset = SkillsTokenDataset(tokenized_encodings_dict, aligned_labels)


In [ ]:
from torch.utils.data import DataLoader, random_split

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)


In [ ]:
from transformers import BertForTokenClassification
from torch.optim import AdamW # Corrected import path for AdamW
import torch

# Initialize BERT for token classification
num_labels = len(label_list)  # O, B-SKILL, I-SKILL
model = BertForTokenClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

# Move model to GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

# Optimizer
optimizer = AdamW(model.parameters(), lr=5e-5)


In [ ]:
from torch.nn import CrossEntropyLoss

epochs = 3  # you can increase later
model.train()

for epoch in range(epochs):
    total_loss = 0
    for batch in train_loader:
        # Move batch to device
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        # Zero gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        logits = outputs.logits

        # Backward pass
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{epochs} - Avg Loss: {avg_loss:.4f}")


In [ ]:
model.eval()


In [ ]:
def compute_accuracy(loader, model):
    correct = 0
    total = 0

    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        with torch.no_grad():
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            predictions = torch.argmax(logits, dim=-1)

        # Flatten predictions and labels
        for pred, label in zip(predictions, labels):
            for p, l in zip(pred, label):
                if l != -100:  # ignore padding tokens
                    total += 1
                    if p == l:
                        correct += 1

    return correct / total * 100  # percentage


In [ ]:
accuracy = compute_accuracy(val_loader, model)
print(f"Validation Accuracy: {accuracy:.2f}%")


In [ ]:
new_text = "I have experience in machine learning"



In [ ]:
# Tokenize the text with the same tokenizer
encoding = tokenizer(new_text.split(),  # split into words
                     is_split_into_words=True,
                     truncation=True,
                     padding="max_length",
                     max_length=128,
                     return_tensors="pt")  # return PyTorch tensors

input_ids = encoding["input_ids"].to(device)
attention_mask = encoding["attention_mask"].to(device)


In [ ]:
model.eval()
with torch.no_grad():
    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    logits = outputs.logits
    predictions = torch.argmax(logits, dim=-1)  # token-level predictions


In [ ]:
pred_labels = [id2label[p.item()] for p in predictions[0]]

# Map tokens to words
# Corrected: encoding.word_ids() already returns the list of word IDs for a single sequence
word_ids = encoding.word_ids()
predicted_skills = []
current_skill = []

for token_idx, label in zip(word_ids, pred_labels):
    if token_idx is None:
        continue
    word = new_text.split()[token_idx]
    if label == "B-SKILL":
        if current_skill:
            predicted_skills.append(" ".join(current_skill))
        current_skill = [word]
    elif label == "I-SKILL":
        current_skill.append(word)
    else:  # "O"
        if current_skill:
            predicted_skills.append(" ".join(current_skill))
            current_skill = []
if current_skill:
    predicted_skills.append(" ".join(current_skill))

print("Predicted Skills:", predicted_skills)


wrong method

In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer

# Initialize MultiLabelBinarizer
mlb = MultiLabelBinarizer()
y = mlb.fit_transform(df['skills_filtered'])

# Save classes for later mapping back to skill names
skill_classes = mlb.classes_

print("Number of unique skills:", len(skill_classes))
print("Example skill classes:", skill_classes[:10])

# Optional: check a row
print("Skills in first row:", df['skills_filtered'].iloc[0])
print("Multi-hot vector:", y[0])


In [ ]:
from transformers import BertTokenizer

# Load BERT tokenizer
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

# Tokenize all texts
tokenized_inputs = tokenizer(
    df['text'].tolist(),
    padding=True,
    truncation=True,
    max_length=128,   # you can adjust based on average text length
    return_tensors="pt"
)


In [ ]:
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
import torch
import numpy as np

# Get the total number of samples
n_samples = len(df)
indices = np.arange(n_samples)

# Split indices into train and validation sets
# We pass 'y' here just to ensure train_test_split gets the correct total length for splitting indices
# and for potential stratification, though stratification based on a multi-label 'y' can be complex.
train_indices, val_indices, _, _ = train_test_split(
    indices,
    y, # Pass y for consistent splitting length and potential stratification
    test_size=0.2,
    random_state=42
)

# Create tokenized inputs and labels for train and validation sets using the split indices
X_train_input_ids = tokenized_inputs['input_ids'][train_indices]
X_train_attention_mask = tokenized_inputs['attention_mask'][train_indices]
y_train_labels = y[train_indices]

X_val_input_ids = tokenized_inputs['input_ids'][val_indices]
X_val_attention_mask = tokenized_inputs['attention_mask'][val_indices]
y_val_labels = y[val_indices]

# Reconstruct the dictionary format for X_train and X_val to match the SkillsDataset constructor
X_train = {'input_ids': X_train_input_ids, 'attention_mask': X_train_attention_mask}
X_val = {'input_ids': X_val_input_ids, 'attention_mask': X_val_attention_mask}

# Assign the split labels to y_train and y_val
y_train = y_train_labels
y_val = y_val_labels


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class SkillsDataset(Dataset):
    def __init__(self, inputs, labels):
        self.input_ids = inputs['input_ids']
        self.attention_mask = inputs['attention_mask']
        self.labels = torch.tensor(labels, dtype=torch.float32)  # BCEWithLogitsLoss needs float

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return {
            "input_ids": self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
            "labels": self.labels[idx]
        }


In [ ]:
train_dataset = SkillsDataset(X_train, y_train)
val_dataset = SkillsDataset(X_val, y_val)


In [ ]:
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8)


In [ ]:
batch["input_ids"]       # BERT token IDs
batch["attention_mask"]  # Attention mask
batch["labels"]          # Multi-hot skill vector


In [ ]:
import torch
from torch import nn
from transformers import BertModel

class SkillClassifier(nn.Module):
    def __init__(self, n_classes):
        super(SkillClassifier, self).__init__()
        self.bert = BertModel.from_pretrained("bert-base-uncased")
        self.dropout = nn.Dropout(0.3)
        self.out = nn.Linear(self.bert.config.hidden_size, n_classes)  # n_classes = number of skills

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.pooler_output  # [CLS] token representation
        dropped = self.dropout(pooled_output)
        logits = self.out(dropped)
        return logits


In [ ]:
from torch.optim import AdamW

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SkillClassifier(n_classes=len(skill_classes))
model = model.to(device)

criterion = nn.BCEWithLogitsLoss()  # Multi-label classification
optimizer = AdamW(model.parameters(), lr=2e-5)


In [ ]:
from tqdm import tqdm

EPOCHS = 3
model.train()

for epoch in range(EPOCHS):
    total_loss = 0
    print(f"\nEpoch {epoch+1}/{EPOCHS}")

    for batch in tqdm(train_loader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()
        logits = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Average Training Loss: {total_loss/len(train_loader):.4f}")


In [ ]:
model.eval()


In [ ]:
all_val_preds = []
all_val_labels = []

with torch.no_grad():  # no gradients needed for inference
    for batch in val_loader:  # using val_loader
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].cpu().numpy()  # keep true labels on CPU

        logits = model(input_ids=input_ids, attention_mask=attention_mask)
        probs = torch.sigmoid(logits).cpu().numpy()  # convert logits to probabilities

        # Convert probabilities to 0/1 using threshold 0.5
        preds = (probs >= 0.5).astype(int)

        all_val_preds.append(preds)
        all_val_labels.append(labels)


In [ ]:
import numpy as np

# Convert list of batches into one big array
all_val_preds = np.vstack(all_val_preds)   # shape: (num_val_samples, num_skills)
all_val_labels = np.vstack(all_val_labels) # shape: (num_val_samples, num_skills)


In [ ]:
import numpy as np

# Compare predictions vs true labels
correct = (all_val_preds == all_val_labels).sum()
total = all_val_labels.size  # total number of labels across all samples
accuracy_percent = correct / total * 100

print(f"Overall Label Accuracy: {accuracy_percent:.2f}%")


In [ ]:
new_title = "Jojo is a new programming language for AI applications"
new_summary = "It simplifies machine learning workflows and integrates with cloud services."

# Combine title + summary
input_text = new_title + " " + new_summary


In [ ]:
encoded = tokenizer(
    input_text,
    truncation=True,
    padding="max_length",
    max_length=128,  # same as training
    return_tensors="pt"
)

input_ids = encoded["input_ids"].to(device)
attention_mask = encoded["attention_mask"].to(device)


In [ ]:
model.eval()  # make sure model is in evaluation mode

with torch.no_grad():  # no gradients needed
    logits = model(input_ids=input_ids, attention_mask=attention_mask)
    probs = torch.sigmoid(logits)  # convert logits to probabilities


In [ ]:
threshold = 0.68
preds = (probs >= threshold).int().cpu().numpy()[0]
predicted_skills = [skill for skill, p in zip(skill_classes, preds) if p == 1]

print("Predicted Skills:", predicted_skills)
